# PoC: Semantic Search

Sistema completo de busca semantica sobre documentos locais.
Demonstracao interativa do pipeline de ingestion e busca.

## Stack
- sentence-transformers/all-MiniLM-L6-v2 (384d embeddings)
- Qdrant (vector database)
- rich (CLI/notebook output)

**Prerequisito:** `docker compose up -d qdrant`

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path

# Executar o pipeline de ingestion
from ingest import ingest

n_chunks = ingest(
    docs_dir=Path('../../data/sample_docs'),
    recreate=True,
)
print(f'Chunks indexados: {n_chunks}')

In [ ]:
from search import search, display_results

# Queries de demonstracao
queries = [
    'Como funciona o algoritmo HNSW para indexacao vetorial?',
    'Quais sao os tipos de quantizacao de vetores?',
    'O que e RAG e como ele evita alucinacoes?',
    'Como os Transformers usam self-attention?',
]

for query in queries:
    results, embed_ms, search_ms = search(query, top_k=3)
    display_results(query, results, embed_ms, search_ms)
    print()

In [ ]:
# Benchmark de latencia
import time
import numpy as np

n_queries = 10
query_benchmark = 'busca semantica com vetores de alta dimensao'

latencias = []
for _ in range(n_queries):
    t0 = time.perf_counter()
    results, _, _ = search(query_benchmark, top_k=5)
    latencias.append((time.perf_counter() - t0) * 1000)

print(f'Benchmark ({n_queries} queries):')
print(f'  P50 latencia: {np.percentile(latencias, 50):.1f}ms')
print(f'  P95 latencia: {np.percentile(latencias, 95):.1f}ms')
print(f'  P99 latencia: {np.percentile(latencias, 99):.1f}ms')
print(f'  Media: {np.mean(latencias):.1f}ms')